##**Video Processing Legacy Notes**
Feb 2022 - Lane Breshears, modified from .py code Alex Day

Mar 2023 - Yisha Tang, modified for ENAMPT

Apr 2024 - Yisha Tang, implemented batch processing for Lexi

October 2024 - Jocelyn Reynolds, implemented upgrades to batch processing to only process new videos and to work around API issues

Use with any microfluidic chip, place the black line over the channel to analyze flow across the channel. Specify number of channels and channel gaps.


##**Pixel Analysis Legacy Notes**

Feb 2022 - Lane Breshears, modified from .py code Alex Day

Mar 2023 - Yisha Tang, modified for eNAMPT: check the example boards before running video analysis; merge multichannel data to one sheet and give threshold-cleaned flow profile

Feb 2024 - Yisha Tang, modified for PFAS: new section of data analysis and visualization to further clean the flow profile, analyze flow rate at adjustable time range.
  - Mar 14: Use the maxiuma variation to locate the initial start if variation doesn't meet the threshold; new funtion - drop the messy data.

Mar 3/20/2024 - Lexi DeFord started modifying this code branch for Micro/Nano plastic detection

Use with any microfluidic chip, place the black line over the channel to analyze flow across the channel. Specify number of channels and channel gaps.

Lexi Notes for restructuring:
-put slope values per chip into dataframe for anova testing

# MasterSandbox Notes
Started Nov 2024 by Lexi DeFord for analyzing all MNP experiments for Round_2

"Batch of Batches"

Running video processing on several folders serially

Creating _Pos.csv files for positional flow profiles

Creating _aic.csv files for after internal control slope values

Creating _Plots.pdf files for visualization of position vs time for each combo

## Setup

**Link CoLab code with your Google Drive**

In [ ]:
!pip uninstall -y opencv-python
!pip install opencv-python==4.10.0.84
!pip uninstall -y os
!pip install kora -q
!pip install pydrive -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# https://drive.google.com/drive/folders/1FWndAYiDDbhP_JWq7UcodWYYF2mCsjqb?usp=sharing

from google.colab import auth
auth.authenticate_user()

import gspread as gs
#from google.auth import default
from apiclient import discovery

from googleapiclient.discovery import build
from google.auth import default

creds, _ = default()
drive_service = build("drive", "v3", credentials=creds)

from tensorflow.python.util.tf_export import get_canonical_name_for_symbol
import numpy as np

#!pip uninstall -y opencv-python
#!pip install opencv-python==4.10.0.84
import cv2
print(cv2.__version__)

#import cv2
import openpyxl
import gspread
from pathlib import Path
#!pip uninstall -y os
import csv
import matplotlib.pyplot as plt
import sys
from google.colab.patches import cv2_imshow #to see images, can be deleted later
import pandas as pd

import gspread_dataframe as gd
import seaborn as sns
import statistics as stat
from matplotlib.backends.backend_pdf import PdfPages



from kora.xattr import get_id
#from pydrive.auth import GoogleAuth
#from pydrive.drive import GoogleDrive
from oauth2client.client import GoogleCredentials
import time
from gspread.exceptions import APIError
gc = gspread.authorize(creds)
# Authenticate and create the PyDrive client.
#gauth = GoogleAuth()
#gauth.credentials = GoogleCredentials.get_application_default()
# Manually build the service to avoid PyDrive trying to look for client_secrets.json
#gauth.service = build('drive', 'v2', credentials=gauth.credentials)
#drive = GoogleDrive(gauth)



import re

**Define Functions**

In [ ]:
from re import split
def defineLane(start, end):
	'''
	Auxilliary function used to correctly determine which pixels to look at in the lane, regardless of whether the pixel
	numbers go from high to low or low to high
	'''
	points = []
	temp = end - start
	if temp < 0:
		for i in range(end, start):
			points.append(i)
	else:
		for i in range(start, end):
			points.append(i)
	return points
#-------------------------------------------------------------------------------
def flowAnalysis(videoName, location, start, end, axes, threshold, rotate_90_clockwise=False):
    '''
	Given a video file, return a list of the flow profile of the moving front. Each index will represent a frame of the
	original video file, and its value will represent how many pixels the wetting front has flowed through

	Parameters:
	"videoName" = the file name of the video file
	"Location" = if the axis is horizontal, then location is the pixel row containing the desired lane, while if the
				axis vertical, then location is the pixel column contained the desired lane
	"start" = the starting pixel in the row/column (dictated by "Location" parameter) of the desired lane
	"end" = the ending pixel in the row/column (dictated by the "Location" parameter) of the desired lane
	"axes" = determines whether the desired flow lane is horizontal or vertical in the video file (MUST BE EITHER "H" OR "V")
	'''

	# NOTE: if you're not seeing any movement in your data, change this threshold value! It represents the brightness change
	# in a pixel within the flow lane that is considered to be high enough to conclude that the moving front has moved to
	# that location.
    threshold = threshold

    possibleAxes = ['H', 'V']
    if axes not in possibleAxes:
        print('Axes parameter not allowed, must be either H or V')
        sys.exit()
    laneLength = defineLane(start, end)

    flowProfile = []
    video = cv2.VideoCapture(videoName)
    #print('Video name: {}'.format(videoName))
    #print('Video length (frames): {}'.format(video.get(cv2.CAP_PROP_FRAME_COUNT)))
    success = True
    imageNumber = 1
    while success:
        # print('Image Number: {}'.format(imageNumber))
        success, image = video.read()
        if success:
            # Rotate 90 degrees clockwise, fixed indentation
            if rotate_90_clockwise == True:
                image = cv2.rotate(image, cv2.ROTATE_90_CLOCKWISE) # jskim edit
            gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
            if imageNumber == 1:
                initialLane = []
                for i in laneLength:
                    triplet = []
                    for j in range(-1, 2):
                        if axes == 'H':
                            triplet.append(gray[location + j, i])
                        elif axes == 'V':
                            triplet.append(gray[i, location + j])
                    initialLane.append(np.average(triplet))

            currentLane = []
            differences = 0
            for i in laneLength:
                triplet = []
                for j in range(-1, 2):
                    if axes == 'H':
                            triplet.append(gray[location + j, i])
                    elif axes == 'V':
                        triplet.append(gray[i, location + j])
                currentLane.append(np.average(triplet))

            for i in range(0, len(laneLength)):
                difference = initialLane[i] - currentLane[i]
                if difference >= threshold:
                    differences += 1

            flowProfile.append(differences)
            imageNumber += 1

    print(f'Maximum flow distance (pixels): {np.amax(flowProfile)}')
    videoFPS = video.get(cv2.CAP_PROP_FPS)
    #print(f'VIDEO FPS IS: {videoFPS}')
    videoDuration = int(video.get(cv2.CAP_PROP_FRAME_COUNT)) / videoFPS
    print('Total velocity (pixel/second): {:2f}'.format(float(np.amax(flowProfile) / videoDuration)))
    print('------------------')

    plt.figure()
    plt.suptitle('Flow Profile (Pixels vs. Frame Number)')
    plt.plot(flowProfile)
    plt.show()
    return flowProfile
#-------------------------------------------------------------------------------
def exampleBounds(videoName, location, start, end, channelnum, channelgap, axes='V',rotate_90_clockwise=False):
  '''
  Given a video file, the function will save a grayscale image depicting where the analysis algorithm would consider
  the lane to be (by using a solid black line over the first frame of the video)

  Parameters:
  "videoName" = the file name of the video file
  "Location" = if the axis is horizontal, then location is the pixel row containing the desired lane, while if the
        axis vertical, then location is the pixel column contained the desired lane
  "start" = the starting pixel in the row/column (dictated by "Location" parameter) of the desired lane
  "end" = the ending pixel in the row/column (dictated by the "Location" parameter) of the desired lane
  "channelnum" = the number of channels in the video file
  "channelgap" = the distance between each channel
  "axes" = determines whether the desired flow lane is horizontal or vertical in the video file (MUST BE EITHER "H" OR "V")
  '''
  possibleAxes = ['H', 'V']
  if axes not in possibleAxes:
    print('Axes parameter not allowed, must be either H or V')
    sys.exit()

  # make sure end value is larger than start value
  # so that start:end can be the index
  if start > end:
    tmp = start
    start = end
    end = tmp

  video = cv2.VideoCapture(videoName)
  print('Video name: {}'.format(videoName))
  print('Video length (frames): {}'.format(video.get(cv2.CAP_PROP_FRAME_COUNT)))
  print('Video FPS (framerate): {}'.format(video.get(cv2.CAP_PROP_FPS)))
  #video.set(cv2.CAP_PROP_POS_FRAMES, frame)
  success, image = video.read()
  if success:
    if rotate_90_clockwise ==True:
      image = cv2.rotate(image, cv2.ROTATE_90_CLOCKWISE) # jskim edit
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    if axes == 'H':
      for i in range(channelnum):
        # draw black lines with given laneshift in one channel
        gray[location+channelgap*i, start:end] = 0
    elif axes == 'V':
      for i in range(channelnum):
        gray[start:end, location+channelgap*i] = 0
    #cv2.imwrite('test bounds.png', gray) # save the test bounds, not necessary
    cv2_imshow(gray)
  else:
    print('Video reading failed, please check the video file name!')
  video.release()
#------------------------------------------------------------------------------
def flowAnalysis_video(videoName, location, start, end, channelnum, channelgap, axes='V', threshold=5, path=None, rotate_90_clockwise=False):

	df_flowProfile = pd.DataFrame()
	for i in list(range(0,channelnum)):
		channel = i + 1
		print(f'channel {channel}')
		if axes == 'H':
			channel_location = location - channelgap*(i) #assuming 1st channel is on bottom of chip
		elif axes == 'V':
			channel_location = location + channelgap*(i) #assuming 1st channel is on far left of chip
		else:
			print('Check which channel you are starting on')

		#print(channel_location)
		#exampleBounds(videoName, channel_location, start, end, axes)
		flowProfile_list = flowAnalysis(videoName, channel_location, start, end, axes, threshold=threshold, rotate_90_clockwise=rotate_90_clockwise) #changed threshold from 9
		channel_name='Ch'+str(i+1)
		df_flowProfile[channel_name]= pd.DataFrame(flowProfile_list)
		#j= next(k for k, value in enumerate(flowProfile_list) if value >= flowLenTh)
		#df_flowProfileTh[channel_name] = pd.DataFrame(flowProfile_list[j:])

	frame = [i+1 for i in range(len(df_flowProfile))]
	df_flowProfile.insert(0, 'frame', frame)
	#A = [i+1 for i in range(len(df_flowProfileTh))]
	#df_flowProfileTh.insert(0, 'frame', A)

	#Save the data to google sheets
	sh = gc.create(str(videoName.stem + '.gsheet'), folder_id=get_id(path))
	ws = sh.get_worksheet(0)# sh.add_worksheet(title='RawData', rows="100", cols="20")
	ws.update_title('RawData')
	ws = sh.worksheet('RawData')#ws = gc.open(str(videoName.name).worksheet('RawData'))
	gd.set_with_dataframe(ws, df_flowProfile)
	return sh

#-------------------------------------------------------------------------------
def FindChannelEnd(data, period = 30, ChannelEnd = 400):
    '''return the 'first' data point of an period with average over 400
    data --> list - 1d [x0, x1, ...]
    Period -->  30 frames, corresponding to s
    '''
    for index in range(len(data) - period):
        mean = stat.mean(data[index : (index + period)])
        if  mean > ChannelEnd:
            return int(index+period/2)
    if FindPlateau(data) != 0:
        return FindPlateau(data)
    return len(data)
#-------------------------------------------------------------------------------
"""def CleanFlowProfile(data, LowerTh = 20, UpperTh = 10):
    '''
    Return the flowrate profile with cleaned beginning and ending
    beginning: cut from 80 pixels when LowerTh=80
    ending: cut til 10 pixels lower than the maximum value when UpperTh=10
    '''
    #a = next(k for k, value in reversed(list(enumerate(data))) if value <= LowerTh)
    a = next(k for k, value in reversed(list(enumerate(data))) if value <= LowerTh)
    b = FindChannelEnd(data[a:], period = 30, ChannelEnd = max(data)-UpperTh)
    plt.figure()
    plt.plot(data)
    Cleaned = data[a:(a+b)]
    plt.plot(Cleaned)
    plt.show()
    plt.close()
    return Cleaned"""
#-------------------------------------------------------------------------------
def FindStartbyVar(data, period = 10, VarTh = 50, FlowDistanceTh = 50):
    var=[]
    for index in range(len(data) - period):
        var.append(stat.variance(data[index : (index + period)]))
    ''' last version
    for k in range(len(var)):
        if var[-k]>VarTh and data[-k-int(1.5*period)]<FlowDistanceTh: break
        cleaned_data=data[-k-int(1.5*period):]
    '''
    for k in range(len(var)-period):
    # -int(period/2) avoid to meet the high peak at the initial
        if var[-k]>50 and data[-k-period]<50:
          cleaned_data=data[-k-period:]
          break
    #print(f"\ncleaned data: {cleaned_data}")
    if k==len(var)-period-1:
      # if above loop end up with no var peak satisfy the standard
      # Use the position of max var as startpoint
        a = var.index(np.max(var))
        if a<10:
            max = np.max(var[a+period:])
            a = var.index(max)
            cleaned_data=data[a:]
    return(cleaned_data)
#------------------------------------------------------------------------------
def Define_Df(df, file_title):
    """ Setting up and filling in a dictionary for channel flow front data per chip """

    #run once per chip to output data in a dictionary per chip with title key
    if file_title.endswith('.mp4'):
        chip_data = {'title': file_title[0:-4]} #removes .mp4 ending
    elif file_title.endswith('.xlsx'):
        chip_data = {'title': file_title[0:-5]} #removes .xlsx ending
    elif file_title.endswith('.gsheet'):
        chip_data = {'title': file_title[0:-7]} #removes .csv ending
    else:
        print("File ending is not .mp4 or .xlsx; check the files.")
    for j in range(len(df.columns)): #iterating over # channels in df
        # this check might be moved to clean**
        data = df.iloc[:,j].tolist()
        data.append('')
        data = data[:data.index('')] #remove the '' at the end of the list
        if data==[]:
            #if data is empty list --> Notify the data is missing
            print(' ch', j+1, ' is missing: ')
        else:
            data=[float(i) for i in data]
        chip_data['Cha ' + str(j)] = data
    return chip_data
#------------------------------------------------------------------------------
def Trim_Data(chip_data, clean_start = True, trim_end = True, channel_start = 50, frame_end = 780):
    """ Trimming the chip data to remove initial data points; was Clean_Data
    """
    clean_chip_data = {'title': chip_data['title']}
    for j in range(len(chip_data)-1): #iterating over clean_chip_data
        current_cha_data = chip_data['Cha ' + str(j)][:]
        #print(f"Cha {str(j)}")
        if clean_start: #Find the initial start
            b = FindStartbyVar(current_cha_data, FlowDistanceTh = channel_start)
            if b[0] < channel_start:
                a=next(k for k, value in reversed(list(enumerate(current_cha_data))) if value <= channel_start)
                current_cha_data=current_cha_data[a:]
        clean_chip_data['T Cha ' + str(j)] = current_cha_data
        if trim_end:
            clean_chip_data['T Cha ' + str(j)] = current_cha_data[:frame_end]
    return clean_chip_data
#------------------------------------------------------------------------------
def Plot_Single_Chip(chip_data):
    """ PLOT FUNCTION FOR A SINGLE CHIP"""

    chip_df = pd.DataFrame(chip_data)
    plt.figure()
    chip_df.plot(xlabel = 'Frame #', ylabel = 'Pixel')
    plt.title(chip_data['title'])
    plt.show()
    plt.close()
    return
#------------------------------------------------------------------------------
def Find_Factor_List(large_data):
    """ To combine replicate chips; expects file names: factor1.factor2.replicate;
    could additionally return r_list if desired """

    #adds to input data dict
    chip_names = sorted(list(large_data.keys()))
    chip_names.remove('title')
    f_list = []
    r_list = []

    #collect list of f1, f2 combinations
    #pfas, concentration_pfas, regent, concentration_regent
    for j in range(len(large_data)-1): # iterating over chips
        #print(chip_names[j])
        #print(re.split(r'[_]', chip_names[j]))
        #[pfas, concentration_pfas, regent, concentration_regent] = chip_names[j].split('.')
        #[pfas, concentration_pfas, regent, concentration_regent, r] = re.split(r'[_]', chip_names[j])
        [pfas, concentration_pfas, regent, concentration_regent, etc, r] = re.split(r'[_]', chip_names[j])
        f_list.append(pfas + "_" + concentration_pfas +"_" + regent + "_"+concentration_regent+"_"+etc) if (pfas + "_" + "concentration_pfas" + regent + "concentration_regent" + etc) not in f_list else 0
        r_list.append(r) if r not in r_list else 0
    return f_list
#------------------------------------------------------------------------------
def Combine_Chip_Data(large_data, f_list):
    """ Linking chip channels together based on f_list entries; uses the first
    combo name length to search for the others! (Have all Plastic and Reagent names
    be the same number of characters!) """

    chip_names = sorted(list(large_data.keys()))
    chip_names.remove('title')
    print("chip_names in Combine_Chip_Data: ")
    print(chip_names)
    combo_len = len(f_list[0])

    print("f_list in Combine_Chip_Data: ")
    #print(f_list)
    #print(f_list[0])
    pixel_by_cha_by_combo = {'title': large_data['title']}
    for combo in f_list:
        curr_combo_data = {}
        curr_c_chip_names = []
        n = 0
        for chip in chip_names: #collecting names of chips that fit the curr combo
            #print(chip)
            #print(combo)
            if chip[0:-2] == combo:
                #print(f"found it: {chip} == {combo}")
                curr_c_chip_names.append(chip)
        #print(curr_c_chip_names)
        n = 0
        for c in curr_c_chip_names:
            chip_entry = large_data[c]
            cha_names = sorted(list(chip_entry.keys()))
            cha_names.remove('title')
            #print(cha_names)
            #print(combo)
            for cha in cha_names:
                curr_combo_data['Cha '+ str(n)] = chip_entry[cha]
                n += 1
        #print(curr_combo_data)
        pixel_by_cha_by_combo[combo] = curr_combo_data
    return pixel_by_cha_by_combo
#------------------------------------------------------------------------------
def Plot_Single_Combo(cha_by_combo_dict):
    """ Plot function for a dict; expects dict of title + combos with dicts of
    channels and pixel data beneath"""

    combo_names = sorted(list(cha_by_combo_dict.keys()))
    combo_names.remove('title')
    for combo in combo_names:
        combo_df = pd.DataFrame(cha_by_combo_dict[combo])
        plt.figure()
        combo_df.plot(xlabel = 'Frame #', ylabel = 'Pixel')
        plt.title(combo)
        plt.show()
        plt.close()
    return
#------------------------------------------------------------------------------
def Step_Calc(data_input, Step = 15, return_table = False, output_edge = 0):
    """ Slope calculation by linear fitting """

    print("Calculating slopes by linear fit of step-sized window")

    FitScale=np.arange(Step, 21*Step, step=Step) # 20 segments for analysis
    data_keys = list(data_input.keys())
    data_keys.remove('title')
    fit_data = pd.DataFrame(columns=['Slope','WindowEdge', 'Combo','PFAS', 'PFAS_Concentration','Reagent', 'Regent_Concentration', 'Etc', 'Channel'])
    for combo in data_keys:
        #print(combo)
        #[f1, r1, f2, r2] = combo.split('_') # expecting keys to be 'xxx.xxx' #파일명에 . 이 있을때 나눔
        [f1, r1, f2, r2, etc] = combo.split('_') # expecting keys to be 'xxx.xxx' #파일명에 . 이 있을때 나눔
        for rep in list(data_input[combo]):
            #print(len(fit_data))
            data = data_input[combo][rep]
            x = np.arange(1, len(data)+1, 1)
            for k in FitScale: #k is the end of the step window
                xx = np.sqrt(x[k-Step:k])
                yy = data[(k-Step):k]
                linearPopt = np.polyfit(xx, yy, 1) #linear fit of the step window, with the [a, b], so [0] is slope
                fit_data.loc[len(fit_data)] = [linearPopt[0], k, combo, f1, r1, f2, r2, etc, int(rep[4:])]
                #fit_data.loc[len(fit_data)] = [linearPopt[0], k, combo, f1[1:], f2, int(rep[4:])]
    if return_table:
        slope_values = fit_data[(fit_data['WindowEdge'] == output_edge)]
        return slope_values
    else:
        return fit_data
#------------------------------------------------------------------------------
def Slim_Step_df(stepped_df, WE = 30, remove_combo = True):
    """ To remove extraneous data from the stepped dataframe """

    slimmed_df = stepped_df[(stepped_df['WindowEdge'] == WE)]
    if remove_combo:
        slimmed_df = slimmed_df.drop(columns = 'Combo')
    return slimmed_df
#------------------------------------------------------------------------------
def Export_CSV(dataframe, filename, filePath):
    """ Export dataframe """

    print(f'\033[92mExporting dataframe to {filename}\033[0m')
    dataframe.to_csv(filePath / filename, index = False)
    return
#------------------------------------------------------------------------------
def CollectInternalControlPerRep(dataframe, channels_per_rep = 4):
  """ Collect Internal Control values, the averaged null condition per repicate """

  print(f'Collecting internal control values')
  #add internal control (IC) column
  dataframe['IC'] = 0.0
  #print(dataframe)
  n_reps = int((max(dataframe["Channel"]) + 1) / channels_per_rep)
  max_zeros =  max(combo.count('0') for combo in dataframe['Combo'].unique())
  def has_two_zeros(combo):
        return combo.count('0') == max_zeros

  # Select only combos that contain the most '0's
  null_combo = [combo for combo in dataframe['Combo'].unique() if has_two_zeros(combo)]
  #print(null_combo)
  for rep in list(range(0,n_reps)):
    for WE in list(dataframe['WindowEdge'].unique()):
      curr_channels = dataframe[
                  (dataframe['Combo'].isin(null_combo)) &
                  (dataframe['WindowEdge'] == WE) &
                  (dataframe['Channel'].between(rep * channels_per_rep, (rep + 1) * channels_per_rep - 1))
              ]
      ic_value = curr_channels['Slope'].mean()
      dataframe.loc[
            (dataframe['WindowEdge'] == WE) & (dataframe['Channel'].between(rep * channels_per_rep, (rep + 1) * channels_per_rep - 1)),
            'IC'
        ] = ic_value
  return dataframe
#------------------------------------------------------------------------------
def SubtractICValues(dataframe):
  """ Subtract IC values """

  print(f'Subtracting IC values')
  dataframe['Adjusted_Slope'] = 0.0
  dataframe['Adjusted_Slope'] = dataframe['Slope'] - dataframe['IC']
  return dataframe
  #----------------------------------------------------------------------------
def get_gsheet_and_xlsx_files(video_files_path):
    """Function to get list of .gsheet and .xlsx files"""
    # Get all files in the directory that are either .gsheet or .xlsx
    data_files = list(video_files_path.iterdir())  # List all files in the directory
    gsheet_files = [file.stem for file in data_files if file.name.endswith('.gsheet')]
    if gsheet_files:
      for index, item in enumerate(gsheet_files, start=1):
          gsheet_files[index - 1] = '.'.join(item.split('.')[:-1])
    xlsx_files = [file.stem for file in data_files if file.name.endswith('.xlsx')]
    return gsheet_files + xlsx_files  # Combine both lists
#------------------------------------------------------------------------------
def BatchProcessVideos(video_files_path, video_files, folderPath, location1 = int(333), start = int(974), end = int(645), channelnum = 4, channelgap = int(142), axes = 'V',rotate_90_clockwise=False ):
  """Main processing code"""
  # video 에서의 측정 선 위치를 결정하는 입력
  #location1 Row/channel initial position for horizontal chip/Verticla chip would be column (moves horizontally)
  #start vertical positioning
  #end vertical positioning
  #channelnum
  #channelgap cellulose jump from channel to chnl in px distance
  #axes H - horizontal, V - vertical

  #video file 위치로 gheet 위치를 받아옴? 만들어옴?
  processed_videos = get_gsheet_and_xlsx_files(video_files_path)
  print(f'Processed videos: {processed_videos}')

  total_videos = len(video_files)

  for index, file in enumerate(video_files, start=1):
      if file.stem in processed_videos:
          print(f'Skipping already processed video: {file}')
          continue  # Skip to the next video

      # 비디오 갯수 알려줌
      print(f'Analyzing video {index} of {total_videos}: {file}')
      file_path = video_files_path / file
      exampleBounds(file_path, location1, start, end, channelnum, channelgap, axes='V',rotate_90_clockwise=rotate_90_clockwise)

      retries = 5
      for attempt in range(retries):
          try:
              sh = flowAnalysis_video(video_files_path / file, location1, start, end, channelnum, channelgap, axes='V', threshold=5, path=video_files_path, rotate_90_clockwise=rotate_90_clockwise)
              #sh = flowAnalysis_video(file, location1, start, end, channelnum, channelgap, axes='V', threshold=5)
              break  # Exit the retry loop if successful
          except APIError as e: # colab 연결 실패 재시도
              print(f"APIError encountered: {e}. Attempt {attempt + 1} of {retries}. Retrying...")
              time.sleep(2 ** attempt)  # Exponential backoff
      else:
          print("Failed to connect to Google Sheets after multiple attempts.")
          continue

      print('------------------------------------------------------------------------')
#-------------------------------------------------------------------------------
def ExportDict(dict_by_cha, filename, filePath):
    df_list = []
    padded_dict = {}  # This will store the padded data

    full_path = Path(filePath) / filename

    for combo in dict_by_cha:
        if combo != 'title':  # Skip the 'title' entry if it exists
            # Find the maximum length among all channels
            #print(dict_by_cha)
            #print(combo)
            max_len = max(len(value) for value in dict_by_cha[combo].values())

            # Pad shorter lists with NaN to match the maximum length
            padded_data = {key: value + [float('nan')] * (max_len - len(value))
                           for key, value in dict_by_cha[combo].items()}

            # Add the padded data to the new dictionary
            padded_dict[combo] = padded_data

            # Convert the padded data into a DataFrame for exporting
            df = pd.DataFrame(padded_data).T  # Create DataFrame with padded data
            df['Combo'] = combo
            df.reset_index(inplace=True)
            df.rename(columns={'index': 'Channel'}, inplace=True)
            df['Combo_Channel'] = df['Combo'] + '.' + df['Channel']
            df = df[['Combo', 'Combo_Channel'] + [col for col in df.columns if col not in ['Combo', 'Combo_Channel']]]
            df_list.append(df)

    # Concatenate all DataFrames to form the final DataFrame
    final_df = pd.concat(df_list, ignore_index=True)
    final_df.to_csv(full_path, index_label='Channel')

    # Print confirmation
    print(f'\033[92mExporting dictionary to {filename}\033[0m')

    # Optionally, return the padded_dict if you want to use it later
    return padded_dict
#------------ 임시
def ExportDict(dict_by_cha, filename, filePath):
    df_list = []
    padded_dict = {}  # This will store the padded data

    full_path = Path(filePath) / filename

    for combo in dict_by_cha:
        if combo != 'title':  # Skip the 'title' entry if it exists
            # Find the maximum length among all channels
            #print(dict_by_cha)
            #print(dict_by_cha[combo].values())
            # Check if dict_by_cha[combo].values() is empty before calling max()
            if dict_by_cha[combo].values():  # Check if there are values in the dictionary
                max_len = max(len(value) for value in dict_by_cha[combo].values())

                # Pad shorter lists with NaN to match the maximum length
                padded_data = {key: value + [float('nan')] * (max_len - len(value))
                               for key, value in dict_by_cha[combo].items()}

                # Add the padded data to the new dictionary
                padded_dict[combo] = padded_data

                # Convert the padded data into a DataFrame for exporting
                df = pd.DataFrame(padded_data).T  # Create DataFrame with padded data
                df['Combo'] = combo
                df.reset_index(inplace=True)
                df.rename(columns={'index': 'Channel'}, inplace=True)
                df['Combo_Channel'] = df['Combo'] + '.' + df['Channel']
                df = df[['Combo', 'Combo_Channel'] + [col for col in df.columns if col not in ['Combo', 'Combo_Channel']]]
                df_list.append(df)
            else:
                print(f"Warning: No data found for combo: {combo}. Skipping this combo.")

    # Concatenate all DataFrames to form the final DataFrame
    final_df = pd.concat(df_list, ignore_index=True)
    final_df.to_csv(full_path, index_label='Channel')

    # Print confirmation
    print(f'\033[92mExporting dictionary to {filename}\033[0m')

    # Optionally, return the padded_dict if you want to use it later
    return padded_dict
#-------------------------------------------------------------------------------
'''def ExportDict(dict_by_cha, filename, filePath):
  df_list = []
  #display(dict_by_cha)
  full_path = filePath / filename
  for combo in dict_by_cha:
    if combo != 'title':
      #cha_names = sorted(list(dict_by_cha[combo].keys()))
      df = pd.DataFrame(dict_by_cha[combo]).T
      df['Combo'] = combo
      df.reset_index(inplace=True)
      df.rename(columns={'index': 'Channel'}, inplace=True)
      df['Combo_Channel'] = df['Combo'] + '.' + df['Channel']
      df = df[['Combo', 'Combo_Channel'] + [col for col in df.columns if col not in ['Combo', 'Combo_Channel']]]
      df_list.append(df)
  final_df = pd.concat(df_list, ignore_index=True)
  final_df.to_csv(full_path, index_label='Channel')
  #print(final_df)
  #print(f'\nSaved dictionary to {full_path}')
  print(f'\033[92mExporting dictionary to {filename}\033[0m')'''
#------------------------------------------------------------------------------
def LoadSheetData(file_id, sheet_name, mime_type, xlsx_path=None):
    if mime_type == 'application/vnd.google-apps.spreadsheet':
        # Handle Google Sheets file
        gsheets = gc.open_by_key(file_id)
        sheets = gsheets.worksheet(sheet_name).get_all_values()
        df = pd.DataFrame(sheets[1:], columns=sheets[0])
    elif mime_type == 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet':
        # Handle .xlsx file (using pandas for simplicity)
        df = pd.read_excel(xlsx_path, sheet_name=sheet_name)
    else:
        raise ValueError("Unsupported file type")
    return df
#------------------------------------------------------------------------------
def GenerateAICPOSPLOTS(FlowDataList, easyName, filepath, sheet_name='RawData', aicFlag=False, posFlag=False, plotpdfFlag=True, plotNowFlag=False):
  """ Function to take list of csv files with raw flow data and apply
  if aicFlag is True: generates after internal control slope (velocity) values and
  export them to a _aic.csv file
  if posFlag is True: trimming before saving the position vs time data (_Pos.csv file)
  if plotpdfFlag is True: collects plots and prints to a _Plots.pdf
  if plotNowFlag is True: plots are printed now in the output
  """

  #collect all data by chip name
  whole_data_dict = {'title': easyName}
  #for file in FlowDataList:
  for i, file in enumerate(FlowDataList, 1):
      retries = 5
      for attempt in range(retries):
        try:
          #df = LoadSheetData(file['id'], sheet_name, file['mimeType'], xlsx_path= os.path.join(os.path.join(filepath, 'Data'), file['title']))
          #df = LoadSheetData(file['id'], sheet_name, file['mimeType'], xlsx_path= filepath / 'Data' / file['title'])
          df = LoadSheetData(file['id'], sheet_name, file['mimeType'], xlsx_path= filepath / 'Data' / file['name'])
          break  # Exit the retry loop if successful
        except APIError as e:
          print(f"APIError encountered: {e}. Attempt {attempt + 1} of {retries}. Retrying...")
          time.sleep(2 ** attempt)  # Exponential backoff
      else:
        print("Failed to connect to Google Sheets after multiple attempts.")
        continue
      df.set_index('frame', inplace=True)
      #print('\nAnalyzing data of: ', file['title'])
      #print(f"\rProcessing file {i} of {len(FlowDataList)}: {file['title']}", end='', flush=True)
      print(f"\rProcessing file {i} of {len(FlowDataList)}: {file['name']}", end='', flush=True)
      #chip_data = Define_Df(df, file['title'])
      chip_data = Define_Df(df, file['name'])
      #Plot_Single_Chip(chip_data)
      tc_data = Trim_Data(chip_data)
      #Plot_Single_Chip(tc_data)
      whole_data_dict[tc_data['title']] = tc_data
  #print(whole_data_dict)
  print('\n')
  f_list = Find_Factor_List(whole_data_dict)
  #print(f_list)
  dict_by_cha = Combine_Chip_Data(whole_data_dict, f_list)
  #print('dict_by_cha:')
  #print(dict_by_cha)

  if plotNowFlag:
    Plot_Single_Combo(dict_by_cha)
  if aicFlag:
    print(f'\033[1mGenerating \033[31m_aic.csv file\033[0m for {easyName}\033[0m')
    #print(dict_by_cha)
    stepped_df = Step_Calc(dict_by_cha) # linear 를 step 으로 변경
    #print(stepped_df)
    ic_df = CollectInternalControlPerRep(stepped_df)
    ic_df = SubtractICValues(ic_df)
    #print(ic_df)
    Export_CSV(ic_df, easyName + '_aic.csv', filepath)
  if posFlag:
    print(f'\033[1mGenerating \033[34m_Pos.csv file\033[0m for {easyName}\033[0m')
    dict_by_cha = ExportDict(dict_by_cha, easyName + '_Pos.csv', filepath)
  if plotpdfFlag: # 이게 유용할것.
    print(f'\033[1mGenerating \033[32m_Plots.pdf\033[0m file for {easyName}\033[0m')
    plot_objects = []  # List to hold plot figures
    combo_names = sorted(list(dict_by_cha.keys()))
    #combo_names.remove('title')
    for combo in combo_names:
        combo_df = pd.DataFrame(dict_by_cha[combo])
        #print(combo_df)
        fig, ax = plt.subplots()
        combo_df.plot(ax=ax, xlabel='Frame #', ylabel='Pixel')
        ax.set_title(combo)
        plot_objects.append(fig)
    with PdfPages(filepath / (easyName + '_Plots.pdf')) as pdf:
        for plot in plot_objects:
            pdf.savefig(plot)  # Save each plot to the PDF
            plt.close(plot)  # Close the figure to avoid memory issues
    #print(f"Plots have been saved to {os.path.join(filepath, easyName + '_Plots.pdf')}")
    print(f'\033[92mExporting plots to {easyName}_Plots.pdf\033[0m')
  return
#-----------------------------------------------------------------------------

def list_sheets_and_xlsx_in_folder(folder_id):
    q = (
        f"'{folder_id}' in parents and trashed=false and ("
        "mimeType='application/vnd.google-apps.spreadsheet' or "
        "mimeType='application/vnd.openxmlformats-officedocument.spreadsheetml.sheet'"
        ")"
    )
    files = []
    page_token = None
    while True:
        resp = drive_service.files().list(
            q=q,
            fields="nextPageToken, files(id, name, mimeType)",
            pageToken=page_token,
            pageSize=1000
        ).execute()
        files.extend(resp.get("files", []))
        page_token = resp.get("nextPageToken")
        if not page_token:
            break
    return files

In [ ]:
""" Script to generate AIC, POS, and Plot files for all experiments if they don't
already have one """
#------------------------------------ 실행전 확인필요 ----------------------------------------------------#
rotate_90_clockwise = False # 비디오 이미지가 회전되어 나올경우 True 로 변경하여 해결
directory_path = Path('/content/drive/MyDrive/data/jiyun/Video') # 파일경로
#------------------------------------  ----------------------------------------------------#
directory_files = directory_path.iterdir()
folders_to_process_aic = []
folders_complete_aic = []
folders_to_process_pos = []
folders_complete_pos = []
folders_complete_plot = []
folders_to_process_plot = []

for folder in directory_files:
    if folder.is_dir() and (folder.name.startswith("2024") or folder.name.startswith("2025") or folder.name.startswith("2026")): # 년도순으로 정리
        #
        experiment_files = list(folder.iterdir())
        aic_files = [file for file in experiment_files if file.name.endswith('_aic.csv')]
        if aic_files:
            folders_complete_aic.append(folder)
        else:
            folders_to_process_aic.append(folder)
        pos_files = [file for file in experiment_files if file.name.endswith('_Pos.csv')]
        if pos_files:
            folders_complete_pos.append(folder)
        else:
            folders_to_process_pos.append(folder)
        plot_files = [file for file in experiment_files if file.name.endswith('_Plots.pdf')]
        if plot_files:
            folders_complete_plot.append(folder)
        else:
            folders_to_process_plot.append(folder)

folders_to_process_aic = sorted(folders_to_process_aic)
folders_to_process_pos = sorted(folders_to_process_pos)
folders_complete_aic = sorted(folders_complete_aic)
folders_complete_pos = sorted(folders_complete_pos)

print(f"AIC: There are {len(folders_to_process_aic)} to process and {len(folders_complete_aic)} complete")
print(f"POS: There are {len(folders_to_process_pos)} to process and {len(folders_complete_pos)} complete")
print(f"PLOT: There are {len(folders_to_process_plot)} to process and {len(folders_complete_plot)} complete")

#exp folder
folders_to_process = list(set(folders_to_process_aic + folders_to_process_pos + folders_to_process_plot))
for i, folder in enumerate(folders_to_process, 1):
    easyName = folder.name
    print(f"\nProcessing folder {i} of {len(folders_to_process)}: \033[1m{easyName}\033[0m")
    data_folder_path = folder / 'Data' # 경로에 Data 를 추가함
    data_files = list(data_folder_path.iterdir())
    # 두 종류의 sheet 로 만듬\
    gsheet_files = [file for file in data_files if file.name.endswith('.gsheet')]
    xlsx_files = [file for file in data_files if file.name.endswith('.xlsx')]
    #csv_files = [file for file in data_files if file.name.endswith('.csv')]
    if len(xlsx_files) > 0:
        gsheet_files.extend(xlsx_files)
    #if len(csv_files) > 0:
    #    gsheet_files.extend(csv_files)
    mp4_files = [file for file in data_files if file.name.endswith('.mp4')]
    gsheet_count = len(gsheet_files)
    mp4_count = len(mp4_files)
    print(f'gsheets: {gsheet_count}    mp4s: {mp4_count}')
    aicFlag = False
    posFlag = False
    plotpdfFlag = False
    #mp4 파일에 비해 sheet 가 부족하면 processing video 시작
    if gsheet_count < mp4_count and mp4_count > 0:
        print(f'Fewer gsheets than mp4s found for {easyName}')
        print(f'Processing videos for {easyName}')
        #Batch Processing
        BatchProcessVideos(data_folder_path, mp4_files, folder,rotate_90_clockwise=rotate_90_clockwise)
   #
    if (gsheet_count > 0 and mp4_count > 0 and gsheet_count == mp4_count) or (gsheet_count > 0 and mp4_count == 0):
        #folderID = str(get_id(data_folder_path))
        # Updated query to include both Google Sheets and .xlsx files
        #q = f"'{folderID}' in parents and (mimeType = 'application/vnd.google-apps.spreadsheet' or mimeType = 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet')"
        #FlowDataList = drive.ListFile({'q': q}).GetList()
        #if folder in folders_to_process_aic:
        #    aicFlag = True
        #if folder in folders_to_process_pos:
        #    posFlag = True
        #if folder in folders_to_process_plot:
        #    plotpdfFlag = True
        #
        #GenerateAICPOSPLOTS(FlowDataList, easyName, data_folder_path.parent, aicFlag=aicFlag, posFlag=posFlag, plotpdfFlag=plotpdfFlag)

        FlowDataList = []

        for f in gsheet_files:  # gsheet_files 안에 xlsx + csv 이미 합쳐둔 상태
            if f.suffix == '.xlsx':
                FlowDataList.append({
                    'id': None,
                    'mimeType': 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet',
                    'title': f.name,  # 나중에 filepath / 'Data' / title 형태로 씀
                })
            elif f.suffix == '.csv':
                FlowDataList.append({
                    'id': None,
                    'mimeType': 'text/csv',
                    'title': f.name,
                })
            # .gsheet는 로컬에 없다고 가정하고 무시

        if folder in folders_to_process_aic:
            aicFlag = True
        if folder in folders_to_process_pos:
            posFlag = True
        if folder in folders_to_process_plot:
            plotpdfFlag = True

        # --- UPDATED SAFE EXECUTION BLOCK ---
        try:
            GenerateAICPOSPLOTS(FlowDataList, easyName, data_folder_path.parent,
                                aicFlag=aicFlag, posFlag=posFlag, plotpdfFlag=plotpdfFlag)
        except ValueError as e:
            if "empty sequence" in str(e):
                print(f"\n[!] DATA ERROR: Sample '{easyName}' has empty channel data.")
                print(f"    Check files in: {data_folder_path}")
                print(f"    This usually means the .csv/.xlsx files exist but are empty or filtered out.")
            else:
                print(f"\n[!] Unexpected ValueError in '{easyName}': {e}")
        except Exception as e:
            print(f"\n[!] Critical Error in '{easyName}': {type(e).__name__}: {e}")
        # ------------------------------------


In [ ]:
""" Collecting _Pos.csv and _aic.csv files into combined POS and AIC csv files """
import os
import shutil
import pandas as pd
from datetime import datetime

# Get the current date
current_date = datetime.today().strftime('%Y%m%d')#양식

# List of folders to search in
directory_path = '/content/drive/MyDrive/data/jiyun/Video'
folder_list_all = os.listdir(directory_path)
folder_list = []
for folder in folder_list_all:
  folder_path = os.path.join(directory_path, folder)
  if os.path.isdir(folder_path) and (folder.startswith("2026") or folder.startswith("2025")):
    folder_list.append(folder_path)

# Target folder where files will be copied
target_folder_pos = '/content/drive/MyDrive/data/jiyun/FlowPath_Data' #
target_folder_aic = '/content/drive/MyDrive/data/jiyun/AIC_Data' #

# List to store dataframes for concatenation
df_list_pos = []
df_list_aic = []

# Loop through each folder
for folder in folder_list:
    # Check if the folder exists
    if os.path.exists(folder):
        # Loop through each file in the directory
        for filename in os.listdir(folder):
            file_path = os.path.join(folder, filename)
            if filename.endswith('_Pos.csv'):
                # Check if the file already exists in the target folder
                target_file_path = os.path.join(target_folder_pos, filename)
                if not os.path.exists(target_file_path):
                    # Copy the file to the target directory
                    shutil.copy(file_path, target_file_path)
                    print(f'Copied: {file_path} to {target_folder_pos}')
            if filename.endswith('_aic.csv'):
                # Check if the file already exists in the target folder
                target_file_path = os.path.join(target_folder_aic, filename)
                if not os.path.exists(target_file_path):
                    # Copy the file to the target directory
                    shutil.copy(file_path, target_file_path)
                    print(f'Copied: {file_path} to {target_folder_aic}')

#After collecting individual files, time to combine them
pos_folder_files = os.listdir(target_folder_pos)
pos_files = [file for file in pos_folder_files if file.endswith('_Pos.csv')]
df_list_pos = []
aic_folder_files = os.listdir(target_folder_aic)
aic_files = [file for file in aic_folder_files if file.endswith('_aic.csv')]
df_list_aic = []
for file in pos_files:
    file_path = os.path.join(target_folder_pos, file)
    df = pd.read_csv(file_path)
    df_list_pos.append(df)
for file in aic_files:
    file_path = os.path.join(target_folder_aic, file)
    df = pd.read_csv(file_path)
    #print(f'file path : {file_path}')
    #print(df)
    df_list_aic.append(df)
combined_df_pos = pd.concat(df_list_pos, ignore_index=True)
combined_df_aic = pd.concat(df_list_aic, ignore_index=True)
combined_csv_path_pos = os.path.join(target_folder_pos, current_date + 'POS_combined.csv')
combined_df_pos.to_csv(combined_csv_path_pos, index=False)
combined_csv_path_aic = os.path.join(target_folder_aic, current_date + 'AIC_combined.csv')
combined_df_aic.to_csv(combined_csv_path_aic, index=False)
print(f"Combined POS CSV saved to: {combined_csv_path_pos}")
print(f"Combined AIC CSV saved to: {combined_csv_path_aic}")


## 비디오 파일 점검용. 이미지 출력이 정상적인지 확인하는 용도

In [ ]:
from pathlib import Path

# Suppose this is the single video you want to process.
# Change this path to your actual file location.
#------------------------------------ 실행전 확인필요 ----------------------------------------------------#
single_video_path = Path('/mnt/data/jiyun/2026_01_15_As_Chi/Data/As_1_Chi_0_1.mp4')
#video_files_path = Path('/mnt/data/jiyun/2026_01_15_As_Chi/Data')
#------------------------------------ 실행전 확인필요 ----------------------------------------------------#
# If you're using rotate_90_clockwise for orientation,
# set this to True if your video needs rotation.
location1 = 313
start = 974
end = 645
channelnum = 4
channelgap = 142
axes = 'V'
rotate_90_clockwise = False

# Now, call BatchProcessVideos directly on just that file.
# Notice that the third parameter (folderPath) is typically
# the folder where we want results saved or from which you read.
gsheet_files = single_video_path.name.endswith('.gsheet')

#BatchProcessVideos(
#    video_files_path=single_video_path.parent,  # Usually the folder containing the MP4
#    video_files=[single_video_path.name],       # We pass just one file
#    folderPath=single_video_path.parent,
#    location1=333,
#    start=974,
#    end=645,
#    channelnum=4,
#    channelgap=142,
#    axes='V',
#    rotate_90_clockwise=rotate_90_clockwise
#)

exampleBounds(single_video_path,
              location=location1,
              start=start,
              end=end,
              channelnum=channelnum,
              channelgap=channelgap,
              axes=axes,
              rotate_90_clockwise=rotate_90_clockwise)
'''
Given a video file, the function will save a grayscale image depicting where the analysis algorithm would consider
the lane to be (by using a solid black line over the first frame of the video)

Parameters:
"videoName" = the file name of the video file
"Location" = if the axis is horizontal, then location is the pixel row containing the desired lane, while if the
      axis vertical, then location is the pixel column contained the desired lane
"start" = the starting pixel in the row/column (dictated by "Location" parameter) of the desired lane
"end" = the ending pixel in the row/column (dictated by the "Location" parameter) of the desired lane
"channelnum" = the number of channels in the video file
"channelgap" = the distance between each channel
"axes" = determines whether the desired flow lane is horizontal or vertical in the video file (MUST BE EITHER "H" OR "V")
'''
retries = 5
for attempt in range(retries):

    sh = flowAnalysis_video(single_video_path, location1, start, end, channelnum,
                            channelgap, axes='V', threshold=5, path=video_files_path,
                            rotate_90_clockwise=rotate_90_clockwise)
    #sh = flowAnalysis_video(file, location1, start, end, channelnum, channelgap, axes='V', threshold=5)
    break  # Exit the retry loop if successful




# 아래는 점검용으로 실행 필요 없음

In [ ]:
""" Collecting _Pos.csv and _aic.csv files into combined POS and AIC csv files """
import os
import shutil
import pandas as pd
from datetime import datetime

# Get the current date
current_date = datetime.today().strftime('%Y%m%d')#양식

# List of folders to search in
directory_path = '/content/drive/MyDrive/PFAS_Yongtae_Junseong/Video'
folder_list_all = os.listdir(directory_path)
folder_list = []
for folder in folder_list_all:
  folder_path = os.path.join(directory_path, folder)
  if os.path.isdir(folder_path) and (folder.startswith("2024") or folder.startswith("2025")):
    folder_list.append(folder_path)

# Target folder where files will be copied
target_folder_pos = '/content/drive/MyDrive/PFAS_Yongtae_Junseong/FlowPath_Data' #
target_folder_aic = '/content/drive/MyDrive/PFAS_Yongtae_Junseong/AIC_Data' #

# List to store dataframes for concatenation
df_list_pos = []
df_list_aic = []

# Loop through each folder
for folder in folder_list:
    # Check if the folder exists
    if os.path.exists(folder):
        # Loop through each file in the directory
        for filename in os.listdir(folder):
            file_path = os.path.join(folder, filename)
            if filename.endswith('_Pos.csv'):
                # Check if the file already exists in the target folder
                target_file_path = os.path.join(target_folder_pos, filename)
                if not os.path.exists(target_file_path):
                    # Copy the file to the target directory
                    shutil.copy(file_path, target_file_path)
                    print(f'Copied: {file_path} to {target_folder_pos}')
            if filename.endswith('_aic.csv'):
                # Check if the file already exists in the target folder
                target_file_path = os.path.join(target_folder_aic, filename)
                if not os.path.exists(target_file_path):
                    # Copy the file to the target directory
                    shutil.copy(file_path, target_file_path)
                    print(f'Copied: {file_path} to {target_folder_aic}')

#After collecting individual files, time to combine them
pos_folder_files = os.listdir(target_folder_pos)
pos_files = [file for file in pos_folder_files if file.endswith('_Pos.csv')]
df_list_pos = []
aic_folder_files = os.listdir(target_folder_aic)
aic_files = [file for file in aic_folder_files if file.endswith('_aic.csv')]
df_list_aic = []
for file in pos_files:
    file_path = os.path.join(target_folder_pos, file)
    df = pd.read_csv(file_path)
    df_list_pos.append(df)
for file in aic_files:
    file_path = os.path.join(target_folder_aic, file)
    df = pd.read_csv(file_path)
    #print(f'file path : {file_path}')
    #print(df)
    df_list_aic.append(df)
combined_df_pos = pd.concat(df_list_pos, ignore_index=True)
combined_df_aic = pd.concat(df_list_aic, ignore_index=True)
combined_csv_path_pos = os.path.join(target_folder_pos, current_date + 'POS_combined.csv')
combined_df_pos.to_csv(combined_csv_path_pos, index=False)
combined_csv_path_aic = os.path.join(target_folder_aic, current_date + 'AIC_combined.csv')
combined_df_aic.to_csv(combined_csv_path_aic, index=False)
print(f"Combined POS CSV saved to: {combined_csv_path_pos}")
print(f"Combined AIC CSV saved to: {combined_csv_path_aic}")


In [ ]:
combined_df_aic[combined_df_aic['Combo'] == 'GENX_0.1_CHI_0_6.0']

In [ ]:
# prompt: combined_df_aic dataframe 에서 'Combo' 에서 'PFOA_100_CHI_1' 몇개인지 확인하기

count_pfoa_100_chi_1 = combined_df_aic[combined_df_aic['Combo'] == 'GENX_10_CHI_0_6.0_1'].shape[0]
print(f"The number of 'PFOA_100_CHI_1' in 'Combo' column is: {count_pfoa_100_chi_1}")


In [ ]:
# prompt: df_list_aic 리스트 안의 dataframe 들에서  'Combo' 에서 'PFOA_100_CHI_1' 몇개인지 확인하기

count_pfoa_100_chi_1_list = []
for df in df_list_aic:
  count = df[df['Combo'] == 'PFOA_10_CHI_1'].shape[0]
  count_pfoa_100_chi_1_list.append(count)

print(f"The number of 'PFOA_100_CHI_1' in 'Combo' column for each dataframe in df_list_aic is: {count_pfoa_100_chi_1_list}")

total_count_pfoa_100_chi_1 = sum(count_pfoa_100_chi_1_list)
print(f"The total number of 'PFOA_100_CHI_1' in 'Combo' column across all dataframes in df_list_aic is: {total_count_pfoa_100_chi_1}")


In [ ]:
df_list_aic[-1]

In [ ]:
from pickle import FALSE
""" To print out 2D grid of graphs for broad overview for data visualization"""


# Step 1: Import necessary libraries
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from google.colab import files

##Think about adding standard deviation bars to graph (even tho they will be v large)



#------------------------------------ 실행전 확인필요 ----------------------------------------------------#
filename = '20250703AIC_combined.csv'
downloadFlag = False
combined_df = pd.read_csv('/content/drive/MyDrive/PFAS_Yongtae_Junseong/AIC_Data/' + filename)
output_file = '/content/drive/MyDrive/PFAS_Yongtae_Junseong/AIC_Data/' + filename[:-3] + 'png'
#------------------------------------  ----------------------------------------------------#

# Step 3: Ensure data has the correct column names
column_headers = ['Slope', 'WindowEdge', 'Combo', 'PFAS', 'Reagent', 'Channel', 'IC', 'Adjusted_Slope']
combined_df = combined_df[column_headers]

#Modify the 'PFAS' column entries that start with 'PC'
combined_df['PFAS'] = combined_df['PFAS'].apply(lambda x: 'SC' + x[1:] if x.startswith('C') else x)

# Step 4: Group the data by 'WindowEdge', 'PFAS', and 'Reagent', and calculate the average Adjusted_Slope
grouped_data = combined_df.groupby(['WindowEdge', 'PFAS', 'Reagent'])['Adjusted_Slope'].mean().reset_index()

# Step 5: Set up the plot styling
sns.set(style="whitegrid")

# Step 6: Get the unique values for 'WindowEdge' and 'PFAS'
window_edges = grouped_data['WindowEdge'].unique()
PFASs = grouped_data['PFAS'].unique()

# Step 7: Calculate the number of subplots (grid size)
num_rows = len(window_edges)
num_cols = len(PFASs)

# Step 8: Create a figure with subplots
fig, axes = plt.subplots(num_rows, num_cols, figsize=(num_cols * 6, num_rows * 6))
fig.subplots_adjust(hspace=0.5, wspace=0.3)  # Adjust space between subplots

# Step 9: Flatten axes array for easy indexing
axes = axes.flatten()

# Step 10: Loop over each combination of 'WindowEdge' and 'PFAS' and create a bar plot for each
for i, (window_edge) in enumerate(window_edges):
    for j, (pfas) in enumerate(PFASs):
        # Get the current subplot
        ax = axes[i * num_cols + j]

        # Filter the data for the current WindowEdge and PFAS
        subset = grouped_data[(grouped_data['WindowEdge'] == window_edge) & (grouped_data['PFAS'] == pfas)]

        # Create a bar plot
        #sns.barplot(x='Reagent', y='Adjusted_Slope', data=subset, palette='viridis', ax=ax)
        # Create a bar plot with standard deviation bars
        sns.barplot(x='Reagent', y='Adjusted_Slope', data=subset, palette='viridis', ax=ax, errorbar='sd', hue='Reagent', legend=False)



        # Set plot title and labels
        ax.set_title(f"WindowEdge = {window_edge}, PFAS = {pfas}")
        ax.set_xlabel('Reagent')
        ax.set_ylabel('Average Adjusted_Slope')
        ax.set_ylim(-5, 5)
        ax.tick_params(axis='x', rotation=45)  # Rotate x-axis labels for readability

# Step 11: Save the grid of plots as an image file

#plt.show()
#sys.exit()
plt.savefig(output_file, bbox_inches='tight')  # Save the figure with tight layout
plt.close()  # Close the figure to avoid displaying it in Colab

if downloadFlag:
    files.download(output_file)
    print(f"Combined plot downloaded")
else:
    print(f"Combined plot saved to: {output_file}")


## 비디오 문제가 있어 파일 한개를 수정할 필요가 있을 경우

In [ ]:
!ls /mnt/data/jiyun/2026_01_15_As_Chi

In [ ]:
data_folder_path = single_video_path / 'Data' # 경로를 Data 로 나눔

In [ ]:
data_folder_path